# SmartFuel Forecast — Machine Learning Model v0

## 1. Project Objective

The objective is to predict whether fuel prices at Spanish petrol stations will increase, decrease, or remain stable.

The model uses historical fuel-price observations collected from the Spanish Ministry's API and stored in AWS S3.

We will:
1. Load and explore the fuel-price data.
2. Prepare historical data for training.
3. Establish a baseline model (no price change).
4. Train a Machine Learning classification model.
5. Evaluate its performance.
6. Save the trained model for deployment on AWS.


In [1]:
import pandas as pd
from pathlib import Path

# Locate the existing dataset
filename = "fuel_prices_2026-11-04_1325.csv"

possible_paths = [
    Path("data/processed") / filename,
    Path("../data/processed") / filename
]

data_path = next(
    (p for p in possible_paths if p.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError("Fuel-price dataset not found.")

# Load the data
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

Dataset shape: (11465, 10)

Columns:
['station_id', 'station_name', 'address', 'municipality', 'province', 'latitude', 'longitude', 'gasoline_95_price', 'diesel_price', 'snapshot_date']

First rows:


,station_id,station_name,address,municipality,province,latitude,longitude,gasoline_95_price,diesel_price,snapshot_date
0,4375,Nº 10.935,"AVENIDA CASTILLA LA MANCHA, 26",Abengibre,ALBACETE,39.211417,-1.539167,1.699,1.839,2026-11-04 13:25:51
1,5122,REPSOL,"CR CM-332, 46,4",Alatoz,ALBACETE,39.100389,-1.346083,1.939,1.999,2026-11-04 13:25:51
2,12054,BP ROMICA,CALLE PRINCIPE DE ASTURIAS (POLÍGONO DE ROMICA...,Albacete,ALBACETE,39.054694,-1.832000,1.759,1.949,2026-11-04 13:25:51
3,13933,PLENERGY,"CALLE FEDERICO GARCIA LORCA, 1",Albacete,ALBACETE,39.000861,-1.849833,1.615,1.729,2026-11-04 13:25:51
4,10765,CARREFOUR,"AVENIDA 1º DE MAYO, S/N",Albacete,ALBACETE,38.985667,-1.868500,1.739,1.899,2026-11-04 13:25:51



## 2. Data Quality and Exploration

Before training the model, we examine the dataset to identify missing values, duplicate records, and the available historical observations.

This step is important because predicting price movements requires multiple observations of the same petrol station over time.


In [2]:

# Convert the snapshot date to datetime
df["snapshot_date"] = pd.to_datetime(
    df["snapshot_date"], errors="coerce"
)

# Convert fuel prices to numeric values
for col in ["gasoline_95_price", "diesel_price"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Explore the dataset
print("Total records:", len(df))
print("Unique stations:", df["station_id"].nunique())
print("Unique snapshot timestamps:", df["snapshot_date"].nunique())

print("\nDate range:")
print("First:", df["snapshot_date"].min())
print("Last:", df["snapshot_date"].max())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nFuel price statistics:")
display(df[["gasoline_95_price", "diesel_price"]].describe())


Total records: 11465
Unique stations: 11465
Unique snapshot timestamps: 1

Date range:
First: 2026-11-04 13:25:51
Last: 2026-11-04 13:25:51

Missing values:
station_id             0
station_name           0
address                0
municipality           0
province               0
latitude               0
longitude              0
gasoline_95_price    566
diesel_price         219
snapshot_date          0
dtype: int64

Duplicate records: 0

Fuel price statistics:


,gasoline_95_price,diesel_price
count,10899.000000,11246.000000
mean,1.708847,1.891002
std,0.097675,0.104352
min,1.345000,1.381000
25%,1.649000,1.799000
50%,1.729000,1.925000
75%,1.779000,1.979000
max,2.299000,2.299000



## 3. Data Preparation

The original dataset contains one row per petrol station, with separate columns for gasoline and diesel prices.

We transform it into a long format, where each row represents one station, one fuel type and one observation timestamp.

This structure is compatible with the SmartFuel data contract and will allow us to compare prices over time.

Missing fuel prices are excluded, as they do not represent valid price observations.


In [3]:

# Define the fuel types according to SmartFuel contracts
fuel_mapping = {
    "gasoline_95_price": "gasolina_95_e5",
    "diesel_price": "gasoleo_a"
}

# Transform from wide format to long format
fuel_df = df.melt(
    id_vars=[
        "station_id",
        "station_name",
        "municipality",
        "province",
        "snapshot_date"
    ],
    value_vars=list(fuel_mapping.keys()),
    var_name="source_fuel",
    value_name="price_eur"
)

# Standardize fuel names
fuel_df["fuel_type"] = fuel_df["source_fuel"].map(fuel_mapping)

# Remove missing or invalid prices
fuel_df = fuel_df.dropna(
    subset=["price_eur", "snapshot_date"]
)
fuel_df = fuel_df[fuel_df["price_eur"] > 0].copy()

# Match SmartFuel's naming conventions
fuel_df = fuel_df.rename(
    columns={"snapshot_date": "ts"}
)
fuel_df = fuel_df.drop(columns=["source_fuel"])

# Make station identifiers strings
fuel_df["station_id"] = fuel_df["station_id"].astype(str)

print("Valid fuel-price observations:", len(fuel_df))

print("\nObservations by fuel type:")
print(fuel_df["fuel_type"].value_counts())

print("\nAvailable timestamps:", fuel_df["ts"].nunique())

display(fuel_df.head())


Valid fuel-price observations: 22145

Observations by fuel type:
fuel_type
gasoleo_a         11246
gasolina_95_e5    10899
Name: count, dtype: int64

Available timestamps: 1


,station_id,station_name,municipality,province,ts,price_eur,fuel_type
0,4375,Nº 10.935,Abengibre,ALBACETE,2026-11-04 13:25:51,1.699,gasolina_95_e5
1,5122,REPSOL,Alatoz,ALBACETE,2026-11-04 13:25:51,1.939,gasolina_95_e5
2,12054,BP ROMICA,Albacete,ALBACETE,2026-11-04 13:25:51,1.759,gasolina_95_e5
3,13933,PLENERGY,Albacete,ALBACETE,2026-11-04 13:25:51,1.615,gasolina_95_e5
4,10765,CARREFOUR,Albacete,ALBACETE,2026-11-04 13:25:51,1.739,gasolina_95_e5



## 4. Baseline Model — No Price Change

Our baseline assumes that the next observed fuel price will be equal to the current price.

- Predicted next price = Current price
- Predicted direction = Flat

This baseline will be used to evaluate whether our Machine Learning model provides better predictions.

The baseline can be generated with one snapshot, but its accuracy cannot be evaluated until future observations are available.


In [4]:

# Create baseline predictions
baseline_df = fuel_df[
    ["station_id", "fuel_type", "ts", "price_eur"]
].copy()

# Assume the next fuel price will remain unchanged
baseline_df["baseline_next_price"] = baseline_df["price_eur"]
baseline_df["baseline_direction"] = "flat"

print("Baseline predictions generated:", len(baseline_df))

display(baseline_df.head())


Baseline predictions generated: 22145


,station_id,fuel_type,ts,price_eur,baseline_next_price,baseline_direction
0,4375,gasolina_95_e5,2026-11-04 13:25:51,1.699,1.699,flat
1,5122,gasolina_95_e5,2026-11-04 13:25:51,1.939,1.939,flat
2,12054,gasolina_95_e5,2026-11-04 13:25:51,1.759,1.759,flat
3,13933,gasolina_95_e5,2026-11-04 13:25:51,1.615,1.615,flat
4,10765,gasolina_95_e5,2026-11-04 13:25:51,1.739,1.739,flat



## 5. Historical Data Preparation

To train the ML model, we need multiple price observations for each petrol station and fuel type.

We will:
1. Sort observations chronologically.
2. Identify the next observed price for each station and fuel type.
3. Calculate the price variation.
4. Classify each variation as "up", "down", or "flat".
5. Prepare the target variable for supervised learning.

Currently, only one snapshot is available. We will calculate the targets once additional historical snapshots have been collected.


In [5]:

import numpy as np

# Prepare observations for historical comparison
history_df = fuel_df[
    ["station_id", "fuel_type", "ts", "price_eur"]
].copy()

# Sort by petrol station, fuel type and timestamp
history_df = history_df.sort_values(
    ["station_id", "fuel_type", "ts"]
)

# Identify the next observed price for each station and fuel
group = history_df.groupby(["station_id", "fuel_type"])

history_df["next_price_eur"] = group["price_eur"].shift(-1)
history_df["next_ts"] = group["ts"].shift(-1)

# Keep observations for which a future price is available
training_df = history_df.dropna(
    subset=["next_price_eur"]
).copy()

# Calculate observed price changes
training_df["price_change_eur"] = (
    training_df["next_price_eur"] - training_df["price_eur"]
).round(3)

# Classify the change
training_df["direction"] = np.select(
    [
        training_df["price_change_eur"] > 0,
        training_df["price_change_eur"] < 0
    ],
    ["up", "down"],
    default="flat"
)

print("Historical observations:", len(history_df))
print("Training examples available:", len(training_df))

if training_df.empty:
    print("Not enough historical snapshots to train the model yet.")
else:
    display(training_df.head())
    print(training_df["direction"].value_counts())


Historical observations: 22145
Training examples available: 0
Not enough historical snapshots to train the model yet.
